# Top2Vec: Temporal Topic Analysis

Analyze topic evolution over time (2000–2025) using **existing best Top2Vec models**
trained on all documents. No retraining needed.

Approach inspired by BERTopic's `topics_over_time()`:
1. Use the single trained model's topic assignments (`doc_top`)
2. Group documents by year using `submitted_date`
3. Compute per-year topic prevalence, coherence, and word evolution

**Advantage over sliced modeling:** Topics are consistent across all years
(same topic IDs), no alignment needed.

In [1]:
import gc
import time
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from top2vec import Top2Vec
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../../data/preprocess")
MODEL_DIR = Path("../../../models/top2vec/tuning")
RESULT_DIR = Path("../../../results/top2vec/temporal")
VERSION = "v1"

# IRBO config
TOP_N_WORDS = 10
RBO_P = 0.9

# Create output directories
for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"Model directory: {MODEL_DIR}")
print(f"Results directory: {RESULT_DIR}")

Subjects: ['cs', 'math', 'physics']
Model directory: ../../../models/top2vec/tuning
Results directory: ../../../results/top2vec/temporal


## Helper Functions

In [3]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load dataset with year column."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year
    return df


def compute_ctfidf_per_year(df, topic_col="topic", text_col="text", top_n=10):
    """
    Compute c-TF-IDF per topic per year.
    
    For each (year, topic), concatenate all documents in that group,
    compute TF (word frequency within the group), then weight by
    IDF (inverse of how many groups contain the word).
    
    Returns:
        topic_words_per_year: dict of {(year, topic_id): [word1, word2, ...]}
    """
    years = sorted(df["year"].unique())
    topics = sorted(df[topic_col].unique())
    
    # Group documents by (year, topic) and concatenate
    groups = df.groupby(["year", topic_col])[text_col].apply(
        lambda x: " ".join(x)
    ).reset_index()
    groups.columns = ["year", "topic", "text"]
    
    # Build vocabulary across all groups
    vectorizer = CountVectorizer(stop_words="english")
    tf_matrix = vectorizer.fit_transform(groups["text"])
    vocab = vectorizer.get_feature_names_out()
    
    # Compute IDF: log(N / df_t) where N = number of groups, df_t = groups containing term
    n_groups = tf_matrix.shape[0]
    df_t = (tf_matrix > 0).sum(axis=0).A1  # document frequency per term
    idf = np.log((n_groups + 1) / (df_t + 1)) + 1  # smoothed IDF
    
    # TF-IDF per group
    tfidf_matrix = tf_matrix.multiply(idf).toarray()
    
    # Normalise rows (L2)
    row_norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1
    tfidf_matrix = tfidf_matrix / row_norms
    
    # Extract top words per (year, topic)
    topic_words_per_year = {}
    for idx, row in groups.iterrows():
        year = row["year"]
        topic = row["topic"]
        scores = tfidf_matrix[idx]
        top_indices = scores.argsort()[-top_n:][::-1]
        top_words = [vocab[i] for i in top_indices if scores[i] > 0]
        topic_words_per_year[(year, topic)] = top_words
    
    return topic_words_per_year


def calculate_coherence_for_words(
    topic_word_lists,
    texts_tokenized,
    dictionary,
) -> float:
    """Calculate C_v coherence given a list of topic word lists."""
    if len(topic_word_lists) == 0:
        return 0.0
    # Filter out empty or too-short topic word lists
    valid_topics = [tw for tw in topic_word_lists if len(tw) >= 2]
    if len(valid_topics) == 0:
        return 0.0
    
    cm = CoherenceModel(
        topics=valid_topics,
        texts=texts_tokenized,
        dictionary=dictionary,
        coherence='c_v',
        processes=1
    )
    return cm.get_coherence()


def rbo(list_1, list_2, p=0.9):
    """Rank-Biased Overlap between two ranked lists."""
    k = min(len(list_1), len(list_2))
    if k == 0:
        return 0.0
    rbo_score = 0.0
    for d in range(1, k + 1):
        set_1 = set(list_1[:d])
        set_2 = set(list_2[:d])
        agreement = len(set_1 & set_2) / d
        rbo_score += (p ** (d - 1)) * agreement
    rbo_score *= (1 - p)
    return rbo_score


def calculate_irbo(topics_words, p=0.9):
    """Calculate mean IRBO diversity across all topic pairs."""
    if len(topics_words) < 2:
        return 0.0
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    return np.mean(irbo_scores)

## Load Models & Data

In [4]:
all_models = {}
all_data = {}
all_years = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")
    
    # Load model
    model_path = MODEL_DIR / subject / "model"
    model = Top2Vec.load(str(model_path))
    all_models[subject] = model
    
    # Load data
    df = load_dataset(subject)
    df["topic"] = model.doc_top  # assign topic from model
    all_data[subject] = df
    
    years = sorted(df["year"].unique())
    all_years[subject] = years
    
    n_topics = model.get_num_topics()
    print(f"  {subject}: {len(df):,} docs, {n_topics} topics, "
          f"{len(years)} years ({years[0]}-{years[-1]})")
    print(f"  Topic sizes: min={model.topic_sizes.min()}, "
          f"max={model.topic_sizes.max()}, "
          f"mean={model.topic_sizes.mean():.0f}")

print(f"\n✅ All subjects loaded")


Loading cs...
  cs: 165,756 docs, 253 topics, 26 years (2000-2025)
  Topic sizes: min=87, max=6473, mean=655

Loading math...
  math: 157,085 docs, 211 topics, 26 years (2000-2025)
  Topic sizes: min=104, max=3997, mean=744

Loading physics...
  physics: 146,311 docs, 204 topics, 26 years (2000-2025)
  Topic sizes: min=107, max=3919, mean=717

✅ All subjects loaded


## Topic Prevalence Over Time

For each year, compute the proportion of documents belonging to each topic.
This shows how topics rise, fall, emerge, and disappear over time.

In [5]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.get_num_topics()

    prevalence_csv = RESULT_DIR / subject / "topic_prevalence.csv"

    print(f"\n{'='*70}")
    print(f"Topic Prevalence: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    prevalence_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs_year = len(year_df)
        topic_counts = year_df["topic"].value_counts()

        for topic_id in range(n_topics):
            count = topic_counts.get(topic_id, 0)
            proportion = count / n_docs_year if n_docs_year > 0 else 0.0

            # Get this topic's top words (from the global model)
            topic_words_arr, _, _ = model.get_topics(n_topics)
            top_words = ", ".join(topic_words_arr[topic_id][:5])

            prevalence_rows.append({
                "subject": subject,
                "year": year,
                "topic_id": topic_id,
                "doc_count": count,
                "total_docs_year": n_docs_year,
                "proportion": round(proportion, 6),
                "top_words": top_words,
            })

        # Summary for this year
        active_topics = (topic_counts > 0).sum()
        top_topic = topic_counts.idxmax()
        top_count = topic_counts.max()
        print(f"  {year}: {n_docs_year:,} docs, {active_topics}/{n_topics} active topics, "
              f"top=T{top_topic} ({top_count} docs)")

    prevalence_df = pd.DataFrame(prevalence_rows)
    prevalence_df.to_csv(prevalence_csv, index=False)
    print(f"\n  Saved to: {prevalence_csv} ({len(prevalence_df)} rows)")


Topic Prevalence: CS (253 topics)
  2000: 488 docs, 106/253 active topics, top=T3 (105 docs)
  2001: 594 docs, 108/253 active topics, top=T3 (57 docs)
  2002: 648 docs, 120/253 active topics, top=T3 (102 docs)
  2003: 825 docs, 127/253 active topics, top=T4 (84 docs)
  2004: 948 docs, 142/253 active topics, top=T3 (75 docs)
  2005: 1,000 docs, 143/253 active topics, top=T3 (61 docs)
  2006: 1,000 docs, 141/253 active topics, top=T3 (73 docs)
  2007: 1,000 docs, 148/253 active topics, top=T1 (60 docs)
  2008: 1,000 docs, 140/253 active topics, top=T1 (55 docs)
  2009: 1,000 docs, 152/253 active topics, top=T1 (65 docs)
  2010: 1,362 docs, 169/253 active topics, top=T1 (77 docs)
  2011: 1,622 docs, 176/253 active topics, top=T1 (107 docs)
  2012: 2,254 docs, 200/253 active topics, top=T1 (139 docs)
  2013: 2,719 docs, 213/253 active topics, top=T1 (159 docs)
  2014: 2,989 docs, 215/253 active topics, top=T1 (171 docs)
  2015: 3,345 docs, 237/253 active topics, top=T1 (174 docs)
  2016: 

## Topic Word Evolution (c-TF-IDF per Year)

Compute c-TF-IDF for each topic at each time point to see how
topic word compositions change over time. This is the core of
BERTopic's `topics_over_time()` approach.

In [6]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    n_topics = model.get_num_topics()

    evolution_csv = RESULT_DIR / subject / "topic_word_evolution.csv"

    print(f"\n{'='*70}")
    print(f"Topic Word Evolution: {subject.upper()}")
    print(f"{'='*70}")

    start = time.time()
    topic_words_per_year = compute_ctfidf_per_year(
        df, topic_col="topic", text_col="text", top_n=TOP_N_WORDS
    )
    elapsed = time.time() - start
    all_topic_words_per_year[subject] = topic_words_per_year

    print(f"  c-TF-IDF computed in {elapsed:.1f}s")
    print(f"  (year, topic) groups: {len(topic_words_per_year)}")

    # Save word evolution
    evolution_rows = []
    for (year, topic_id), words in sorted(topic_words_per_year.items()):
        evolution_rows.append({
            "subject": subject,
            "year": year,
            "topic_id": topic_id,
            "top_words": ", ".join(words),
        })

    evolution_df = pd.DataFrame(evolution_rows)
    evolution_df.to_csv(evolution_csv, index=False)
    print(f"  Saved to: {evolution_csv}")

    # Show example: topic 0 across a few years
    print(f"\n  Example — Topic 0 word evolution:")
    for year in [2000, 2005, 2010, 2015, 2020, 2025]:
        key = (year, 0)
        if key in topic_words_per_year:
            words = ", ".join(topic_words_per_year[key][:5])
            print(f"    {year}: {words}")


Topic Word Evolution: CS
  c-TF-IDF computed in 29.5s
  (year, topic) groups: 5050
  Saved to: ../../../results/top2vec/temporal/cs/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: robot, ccgolog, congolog, multiagent, robots
    2005: ansprolog, control, robot, receding, manipulators
    2010: idiotypic, robot, rgt, torque, robots
    2015: control, robot, robots, planning, locomotion
    2020: learning, robot, reinforcement, policy, control
    2025: robot, learning, policy, rl, control

Topic Word Evolution: MATH
  c-TF-IDF computed in 18.9s
  (year, topic) groups: 5210
  Saved to: ../../../results/top2vec/temporal/math/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: bkm, boundary, shock, equation, reynolds
    2005: numerical, wrong, fd, td, richtmyer
    2010: method, numerical, error, element, methods
    2015: numerical, mesh, method, element, galerkin
    2020: numerical, method, galerkin, element, error
    2025: numerical, metho

## Per-Year Coherence & IRBO

For each year, compute coherence and IRBO using that year's c-TF-IDF
topic word lists. This measures how well-defined and diverse the topics
are at each point in time.

In [7]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.get_num_topics()
    topic_words_per_year = all_topic_words_per_year[subject]

    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"

    print(f"\n{'='*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    metrics_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs = len(year_df)
        texts_tokenized = [text.split() for text in year_df["text"].tolist()]
        dictionary = Dictionary(texts_tokenized)

        # Get topic word lists for this year
        active_topics = sorted(year_df["topic"].unique())
        year_topic_words = []
        for tid in active_topics:
            key = (year, tid)
            if key in topic_words_per_year and len(topic_words_per_year[key]) >= 2:
                year_topic_words.append(topic_words_per_year[key])

        # Coherence
        coherence = calculate_coherence_for_words(
            year_topic_words, texts_tokenized, dictionary
        )

        # IRBO
        irbo_mean = calculate_irbo(year_topic_words, p=RBO_P)

        # Topic Quality = harmonic mean
        if coherence + irbo_mean > 0:
            topic_quality = 2 * coherence * irbo_mean / (coherence + irbo_mean)
        else:
            topic_quality = 0.0

        n_active = len(active_topics)

        print(f"  {year}: {n_docs:,} docs, {n_active} active topics | "
              f"Quality={topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_mean:.4f})")

        metrics_rows.append({
            "subject": subject,
            "year": year,
            "num_docs": n_docs,
            "num_topics_total": n_topics,
            "num_topics_active": n_active,
            "coherence_cv": round(coherence, 6),
            "irbo_mean": round(irbo_mean, 6),
            "topic_quality": round(topic_quality, 6),
        })

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(metrics_csv, index=False)
    print(f"\n  Saved to: {metrics_csv}")


Per-Year Metrics: CS (253 topics)
  2000: 488 docs, 106 active topics | Quality=0.7290 (C=0.5739, IRBO=0.9988)
  2001: 594 docs, 108 active topics | Quality=0.6924 (C=0.5300, IRBO=0.9982)
  2002: 648 docs, 120 active topics | Quality=0.7164 (C=0.5586, IRBO=0.9983)
  2003: 825 docs, 127 active topics | Quality=0.6582 (C=0.4909, IRBO=0.9988)
  2004: 948 docs, 142 active topics | Quality=0.6737 (C=0.5082, IRBO=0.9991)
  2005: 1,000 docs, 143 active topics | Quality=0.6815 (C=0.5172, IRBO=0.9989)
  2006: 1,000 docs, 141 active topics | Quality=0.6937 (C=0.5314, IRBO=0.9989)
  2007: 1,000 docs, 148 active topics | Quality=0.6830 (C=0.5189, IRBO=0.9990)
  2008: 1,000 docs, 140 active topics | Quality=0.6742 (C=0.5088, IRBO=0.9989)
  2009: 1,000 docs, 152 active topics | Quality=0.6632 (C=0.4963, IRBO=0.9989)
  2010: 1,362 docs, 169 active topics | Quality=0.6489 (C=0.4805, IRBO=0.9988)
  2011: 1,622 docs, 176 active topics | Quality=0.6632 (C=0.4965, IRBO=0.9984)
  2012: 2,254 docs, 200 act

## Topic Trends: Emerging, Growing, and Declining Topics

Identify which topics are trending up, trending down,
or stable over the full time period.

In [8]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.get_num_topics()

    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    print(f"\n{'='*70}")
    print(f"Topic Trends: {subject.upper()}")
    print(f"{'='*70}")

    topic_words_arr, _, _ = model.get_topics(n_topics)

    trend_rows = []

    for topic_id in range(n_topics):
        topic_df = df[df["topic"] == topic_id]
        if len(topic_df) == 0:
            continue

        # Per-year proportions for this topic
        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()

        proportions = (year_counts / total_per_year).fillna(0)

        # First and last year this topic appears
        first_year = year_counts.index.min()
        last_year = year_counts.index.max()

        # Early vs late proportion (first 5 years vs last 5 years)
        topic_years = sorted(year_counts.index)  # only years where topic has docs
        early_years = topic_years[:5]
        late_years = topic_years[-5:]

        early_mean = proportions[early_years].mean()
        late_mean = proportions[late_years].mean()

        if early_mean > 0:
            trend_ratio = late_mean / early_mean
        elif late_mean > 0:
            trend_ratio = float('inf')
        else:
            trend_ratio = 1.0

        if trend_ratio > 2.0:
            trend_label = "GROWING"
        elif trend_ratio < 0.5:
            trend_label = "DECLINING"
        else:
            trend_label = "STABLE"

        top_words = ", ".join(topic_words_arr[topic_id][:5])

        trend_rows.append({
            "subject": subject,
            "topic_id": topic_id,
            "top_words": top_words,
            "total_docs": len(topic_df),
            "first_year": first_year,
            "last_year": last_year,
            "early_proportion": round(early_mean, 6),
            "late_proportion": round(late_mean, 6),
            "trend_ratio": round(trend_ratio, 4),
            "trend": trend_label,
        })

    trends_df = pd.DataFrame(trend_rows)
    trends_df.to_csv(trends_csv, index=False)

    # Summary
    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])
    print(f"  Total topics: {n_topics}")
    print(f"  Growing:  {growing} topics")
    print(f"  Stable:   {stable} topics")
    print(f"  Declining: {declining} topics")

    print(f"\n  Top 5 GROWING topics:")
    growing_df = trends_df[trends_df["trend"] == "GROWING"].copy()
    growing_df["trend_ratio_num"] = pd.to_numeric(growing_df["trend_ratio"], errors="coerce")
    for _, row in growing_df.nlargest(5, "trend_ratio_num").iterrows():
        print(f"    T{row['topic_id']}: {row['top_words']} "
              f"(ratio={row['trend_ratio']}, early={row['early_proportion']:.4f}→late={row['late_proportion']:.4f})")

    print(f"\n  Top 5 DECLINING topics:")
    declining_df = trends_df[trends_df["trend"] == "DECLINING"].copy()
    declining_df["trend_ratio_num"] = pd.to_numeric(declining_df["trend_ratio"], errors="coerce")
    for _, row in declining_df.nsmallest(5, "trend_ratio_num").iterrows():
        print(f"    T{row['topic_id']}: {row['top_words']} "
              f"(ratio={row['trend_ratio']}, early={row['early_proportion']:.4f}→late={row['late_proportion']:.4f})")

    print(f"\n  Saved to: {trends_csv}")


Topic Trends: CS
  Total topics: 253
  Growing:  90 topics
  Stable:   101 topics
  Declining: 62 topics

  Top 5 GROWING topics:
    T27: federated, learns, efficientnet, supervised, adversarially (ratio=18.0113, early=0.0005→late=0.0098)
    T6: multimodal, captioning, multimodality, deepmind, deeplabv (ratio=12.4732, early=0.0013→late=0.0157)
    T48: imagenet, autoencoders, multimodal, inception, deepmind (ratio=11.6799, early=0.0006→late=0.0070)
    T11: graphon, hypergraph, graphs, networks, subgraph (ratio=11.4738, early=0.0012→late=0.0134)
    T2: imagenet, deeplabv, convnets, cnn, inceptionv (ratio=10.5238, early=0.0027→late=0.0279)

  Top 5 DECLINING topics:
    T252: spreadsheets, spreadsheet, excels, excelling, excel (ratio=0.0151, early=0.0091→late=0.0001)
    T145: relaying, relay, relays, transmit, communications (ratio=0.0495, early=0.0084→late=0.0004)
    T213: authorization, privileges, delegation, implementations, security (ratio=0.052, early=0.0104→late=0.0005)
   

## Evolution Summary

In [9]:
for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        print(f"{subject}: missing metrics, skipping")
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)

    model = all_models[subject]
    n_topics = model.get_num_topics()

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    summary_data = {
        "subject": subject,
        "num_topics": n_topics,
        "num_years": len(metrics_df),
        "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
        "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
        "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
        "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
        "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
        "quality_std": round(metrics_df["topic_quality"].std(), 6),
        "topics_growing": growing,
        "topics_stable": stable,
        "topics_declining": declining,
    }
    summary_df = pd.DataFrame([summary_data])
    summary_csv = RESULT_DIR / subject / "evolution_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"\n  {subject.upper()} summary saved to: {summary_csv}")


  CS summary saved to: ../../../results/top2vec/temporal/cs/evolution_summary.csv

  MATH summary saved to: ../../../results/top2vec/temporal/math/evolution_summary.csv

  PHYSICS summary saved to: ../../../results/top2vec/temporal/physics/evolution_summary.csv


## Final Results

In [10]:
print("\n" + "=" * 110)
print("TOP2VEC TEMPORAL ANALYSIS: FINAL RESULTS")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        print(f"\n{subject.upper()}: No results found")
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)

    model = all_models[subject]
    n_topics = model.get_num_topics()

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    sep = chr(9472)
    print(f"\n{sep*60}")
    print(f"  Subject:        {subject.upper()}")
    print(f"  Num topics:     {n_topics}")
    print(f"  Years:          {len(metrics_df)}")
    print(f"  Coherence:      {metrics_df['coherence_cv'].mean():.4f} +/- {metrics_df['coherence_cv'].std():.4f}")
    print(f"  IRBO:           {metrics_df['irbo_mean'].mean():.4f} +/- {metrics_df['irbo_mean'].std():.4f}")
    print(f"  Topic Quality:  {metrics_df['topic_quality'].mean():.4f} +/- {metrics_df['topic_quality'].std():.4f}")
    print(f"  Trends:         ↑{growing} growing, →{stable} stable, ↓{declining} declining")
    print(f"{sep*60}")

print("\n" + "=" * 110)
print("Per-Year Details:")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    if not metrics_csv.exists():
        continue

    metrics_df = pd.read_csv(metrics_csv)

    print(f"\n{subject.upper()}:")
    print(metrics_df[["year", "num_docs", "num_topics_active",
                     "coherence_cv", "irbo_mean", "topic_quality"]].to_string(index=False))
    print()


TOP2VEC TEMPORAL ANALYSIS: FINAL RESULTS

────────────────────────────────────────────────────────────
  Subject:        CS
  Num topics:     253
  Years:          26
  Coherence:      0.5067 +/- 0.0493
  IRBO:           0.9963 +/- 0.0034
  Topic Quality:  0.6704 +/- 0.0431
  Trends:         ↑90 growing, →101 stable, ↓62 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        MATH
  Num topics:     211
  Years:          26
  Coherence:      0.4656 +/- 0.0453
  IRBO:           0.9946 +/- 0.0040
  Topic Quality:  0.6330 +/- 0.0412
  Trends:         ↑56 growing, →121 stable, ↓34 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        PHYSICS
  Num topics:     204
  Years:          26
  Coherence:      0.4965 +/- 0.0459
  IRBO:           0.9969 +/- 0.0016
  Topic Quality:  0.6616 +/- 0.0402
  Trends:  